# Compare TOS10 equation of state formulation with the equation of state of Chen and Millero 1986

In [1]:
#import libraries needed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import gsw
from math import sqrt
from scipy.optimize import curve_fit

**define the function computing the density using the equation of state of Chen and Millero 1986**

In [2]:
def rho_chen_and_millero(S,t,p):
    """
    Calculate density using Chen and Millero (1986) formulation

    Parameters
    ----------
    S : salinity
    t : in situ temperature (t68)
    p : pressure (bar)
    
    Returns
    -------
    rho : density (kg/m3)
    """
    S=np.array(S) ; t=np.array(t) ; p=np.array(p)
    rho_0=0.9998395+6.7914*10**(-5)*t-9.0894*10**(-6)*t**2+1.0171*10**(-7)*t**3-1.2846*10**(-9)*t**4+1.1592*10**(-11)*t**5-5.0125*10**(-14)*t**6+(8.181*10**(-4)-3.85*10**(-6)*t+4.96*10**(-8)*t**2)*S
    K=19652.17+148.113*t-2.293*t**2+1.256*10**(-2)*t**3-4.18*10**(-5)*t**4+(3.2726-2.147*10**(-4)*t+1.128*10**(-4)*t**2)*p+(53.238-0.313*t+5.728*10**(-3)*p)*S
    rho=rho_0/(1-p/K)
    return rho*1000

**define some variables**

In [3]:
#reference salinity, temperature and in situ density
S0=0.5 ; T0=4 ; t0=gsw.conversions.t_from_CT(S0,T0,0)
print(gsw.density.rho_t_exact(S0,t0,0))
rho0=1000.38

1000.3763825587874


In [4]:
#define our ranges of interest
taille=550
SA_min=0 ; SA_max=0.6 #g/kg
CT_min=0 ; CT_max=10 #°C

SA=np.linspace(SA_min,SA_max,taille)
CT=np.linspace(CT_min,CT_max,taille)
z=np.array([-10,-350,-710]) # m
p=-z*rho0*9.80665*10**(-4)

rho_freshwater=np.ones((len(SA),len(CT),len(p)))*np.nan
rho_gswt=np.ones((len(SA),len(CT),len(p)))*np.nan
rho_gswCT=np.ones((len(SA),len(CT),len(p)))*np.nan
rho_polyTEOS10=np.ones((len(SA),len(CT),len(p)))*np.nan

**compute densities**

In [5]:
ip=0
while ip<len(p):
    iSA=0
    while iSA<len(SA):
        #conversions
        P=p[ip]*0.1 #pressure from dbar to bar
        t=gsw.conversions.t_from_CT(SA[iSA],CT,p[ip]) #in situ temperature (t90) from Conservative Temperature
        t68=t*1.00024 #in situ temperature t68 from in situ temperature t90
        #calculs
        rho_freshwater[iSA,:,ip]=rho_chen_and_millero(SA[iSA],t68,P) #SA (g/kg) ; t in situ temperature t68 (°C) ; p gauge pressure (bar)
        rho_gswt[iSA,:,ip]=gsw.density.rho_t_exact(SA[iSA],t,p[ip]) #SA (g/kg) ; t in situ temperature t90 (°C) ; p pressure (dbar)
        iSA+=1
    ip+=1

**Comparisons**

In [6]:
#difference TEOS10 and Chen and Millero 1986
diff_gswt=rho_gswt-rho_freshwater

In [8]:
#display some RMSE

print("RMSE at "+str(-z[0])+"m")
RMSE_gswt0=sqrt(np.mean(diff_gswt[:,:,0]**2))
print("mean density :", np.mean(rho_freshwater[:,:,0])) #to compare the order of magnitude with the RMSE
print("RMSE :",RMSE_gswt0)

print("RMSE at "+str(-z[1])+"m")
RMSE_gswt1=sqrt(np.mean(diff_gswt[:,:,1]**2))
print("mean density :",np.mean(rho_freshwater[:,:,1])) #to compare the order of magnitude with the RMSE
print("RMSE :",RMSE_gswt1)

print("RMSE at "+str(-z[2])+"m")
RMSE_gswt2=sqrt(np.mean(diff_gswt[:,:,2]**2))
print("mean density :",np.mean(rho_freshwater[:,:,2]))#to compare the order of magnitude with the RMSE
print("RMSE :",RMSE_gswt2)

RMSE at 10m
mean density : 1000.1972712117928
RMSE : 0.0030910649579046507
RMSE at 350m
mean density : 1001.8339877498126
RMSE : 0.0029366821865927117
RMSE at 710m
mean density : 1003.5535981310201
RMSE : 0.0027604264658259822
